In [ ]:
from google.colab import files

uploaded = files.upload()  # ← this opens the file picker


Saving nigerian_market_prices_assignment.csv to nigerian_market_prices_assignment (1).csv
Saving nigerian_market_transactions_assignment.csv to nigerian_market_transactions_assignment (1).csv


In [ ]:
!ls -lh



total 268K
-rw-r--r-- 1 root root  63K Feb 25 11:14 'nigerian_market_prices_assignment (1).csv'
-rw-r--r-- 1 root root  63K Feb 25 11:12  nigerian_market_prices_assignment.csv
-rw-r--r-- 1 root root  67K Feb 25 11:14 'nigerian_market_transactions_assignment (1).csv'
-rw-r--r-- 1 root root  67K Feb 25 11:12  nigerian_market_transactions_assignment.csv
drwxr-xr-x 1 root root 4.0K Jan 16 14:24  sample_data


In [ ]:
import pandas as pd

# Use the names exactly as shown in your upload message
prices = pd.read_csv("nigerian_market_prices_assignment (1).csv")
transactions = pd.read_csv("nigerian_market_transactions_assignment (1).csv")

# Quick check
print("Prices shape:", prices.shape)
print("Transactions shape:", transactions.shape)

print("\nPrices first 3 rows:")
print(prices.head(3))

print("\nTransactions first 3 rows:")
print(transactions.head(3))

Prices shape: (1000, 8)
Transactions shape: (1000, 9)

Prices first 3 rows:
  market_id    market_name  state product   price    unit        date  \
0    MKT001  Bodija Market    Oyo   Beans  ₦5,770      Kg  01-02-2024   
1    MKT003  Ogbete Market  Enugu     Yam  ₦2,534  basket  07/02/2024   
2    MKT001  Bodija Market    Oyo     Yam     NaN    50kg  22/02/2024   

   trader_name  
0  mama nkechi  
1        bello  
2  alhaji musa  

Transactions first 3 rows:
  transaction_id market_id trader_id  trader_name   product  quantity  \
0        TXN0001    MKT005     TR005  ALHAJI MUSA      Gari         8   
1        TXN0002    MKT004     TR001       SADIYA     Beans        14   
2        TXN0003    MKT002     TR006  ALHAJI MUSA  Tomatoes         8   

   revenue payment_method  transaction_date  
0    86160       Transfer  2024-02-03 13:53  
1   492226            POS  2024-01-05 13:46  
2   283392       Transfer  2024-01-07 16:53  


In [ ]:
print("=== PRICES INFO ===")
print(prices.info())
print("\nMissing values in prices:\n", prices.isnull().sum())

print("\n=== TRANSACTIONS INFO ===")
print(transactions.info())
print("\nMissing values in transactions:\n", transactions.isnull().sum())

=== PRICES INFO ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   market_id    1000 non-null   object
 1   market_name  1000 non-null   object
 2   state        1000 non-null   object
 3   product      1000 non-null   object
 4   price        925 non-null    object
 5   unit         1000 non-null   object
 6   date         1000 non-null   object
 7   trader_name  1000 non-null   object
dtypes: object(8)
memory usage: 62.6+ KB
None

Missing values in prices:
 market_id       0
market_name     0
state           0
product         0
price          75
unit            0
date            0
trader_name     0
dtype: int64

=== TRANSACTIONS INFO ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   trans

In [ ]:
# Convert dates – tell pandas to handle mixed formats and invalid values
prices['week_start_date'] = pd.to_datetime(prices['week_start_date'], errors='coerce')
transactions['transaction_date'] = pd.to_datetime(transactions['transaction_date'], errors='coerce')

print("After date conversion:")
print("Prices week_start_date dtype:", prices['week_start_date'].dtype)
print("Transactions transaction_date dtype:", transactions['transaction_date'].dtype)

KeyError: 'week_start_date'

In [ ]:
print("Column names in prices:", prices.columns.tolist())

Column names in prices: ['market_id', 'market_name', 'state', 'product', 'price', 'unit', 'date', 'trader_name']


In [ ]:
# Make sure dates are datetime
prices['date'] = pd.to_datetime(prices['date'], errors='coerce')
transactions['transaction_date'] = pd.to_datetime(transactions['transaction_date'], errors='coerce')

# Create a week-start column from transaction_date (to match weekly prices)
transactions['week_start'] = transactions['transaction_date'].dt.to_period('W').dt.start_time

# Merge on product + approximate week start date
merged = pd.merge(
    transactions,
    prices,
    left_on=['product', 'week_start'],
    right_on=['product', 'date'],           # ← changed to 'date'
    how='left'
)

# Fill remaining missing prices with median per product
merged['price'] = merged.groupby('product')['price'].transform(lambda x: x.fillna(x.median()))

# Quick check
print("Merged shape:", merged.shape)
print("\nFirst 5 rows of merged:")
print(merged.head())
print("\nMissing price after fill:", merged['price'].isnull().sum())

TypeError: Cannot convert ['3229' '₦6,689' nan nan nan nan nan nan nan nan nan nan nan '3229'
 '₦6,689' nan nan nan nan nan nan nan nan '3229' '₦6,689' nan nan nan nan
 nan nan nan nan nan nan '3229' '₦6,689' nan nan nan nan nan nan nan
 '3229' '₦6,689' nan nan nan nan nan nan nan nan nan '3229' '₦6,689' nan
 '3229' '₦6,689' nan nan '3229' '₦6,689' nan nan nan nan nan nan nan nan
 '3229' '₦6,689' nan nan nan nan nan nan nan nan nan nan nan nan '3229'
 '₦6,689' nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 '3229' '₦6,689' nan nan nan nan nan nan nan nan nan nan '3229' '₦6,689'
 nan nan nan nan nan '3229' '₦6,689' nan nan nan nan nan nan nan nan nan
 nan nan nan '3229' '₦6,689' nan '3229' '₦6,689' nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan '3229' '₦6,689' nan nan nan
 nan nan nan nan nan nan nan nan '3229' '₦6,689' nan nan nan nan nan nan
 nan nan '3229' '₦6,689' nan nan nan 

In [ ]:
print("Price column dtype:", prices['price'].dtype)
print("\nFirst 10 price values:\n", prices['price'].head(10))
print("\nUnique sample values:\n", prices['price'].unique()[:20])

Price column dtype: object

First 10 price values:
 0     ₦5,770
1     ₦2,534
2        NaN
3        NaN
4      -2616
5     ₦6,166
6     ₦5,367
7      -1901
8    ₦43,890
9       6292
Name: price, dtype: object

Unique sample values:
 ['₦5,770' '₦2,534' nan '-2616' '₦6,166' '₦5,367' '-1901' '₦43,890' '6292'
 '₦112' '₦-915' '643' '-3639' '₦5,129' '3063' '₦1,059' '₦-1,173' '40391'
 '2384' '6496']


In [ ]:
# Remove ₦ symbol and commas, then convert to float
prices['price'] = prices['price'].astype(str)  # make sure all are strings first

# Remove currency symbol and commas
prices['price'] = prices['price'].str.replace('₦', '', regex=False)
prices['price'] = prices['price'].str.replace(',', '', regex=False)

# Convert to numeric (turn errors → NaN)
prices['price'] = pd.to_numeric(prices['price'], errors='coerce')

print("After cleaning:")
print("Price dtype:", prices['price'].dtype)
print("Missing after conversion:", prices['price'].isnull().sum())
print(prices['price'].describe())

After cleaning:
Price dtype: float64
Missing after conversion: 75
count      925.000000
mean     11446.982703
std      17886.040373
min      -3791.000000
25%        273.000000
50%       3293.000000
75%       7301.000000
max      49911.000000
Name: price, dtype: float64


In [ ]:
# Re-create week_start (if needed)
transactions['week_start'] = transactions['transaction_date'].dt.to_period('W').dt.start_time

# Merge again
merged = pd.merge(
    transactions,
    prices,
    left_on=['product', 'week_start'],
    right_on=['product', 'date'],
    how='left'
)

# Now fill missing prices (this line should now succeed)
merged['price'] = merged.groupby('product')['price'].transform(lambda x: x.fillna(x.median()))

# Check result
print("Merged shape:", merged.shape)
print("\nMissing price after fill:", merged['price'].isnull().sum())
print(merged[['product', 'price', 'transaction_date', 'date']].head(10))

Merged shape: (1018, 17)

Missing price after fill: 397
    product    price    transaction_date       date
0      Gari      NaN 2024-02-03 13:53:00        NaT
1     Beans   3229.0 2024-01-05 13:46:00 2024-01-01
2     Beans   6689.0 2024-01-05 13:46:00 2024-01-01
3  Tomatoes      NaN 2024-01-07 16:53:00        NaT
4     Beans   4959.0 2024-03-02 10:14:00        NaT
5      Rice  43340.0 2024-01-27 14:37:00        NaT
6       Yam   3737.0 2024-02-29 18:50:00        NaT
7      Rice  43340.0 2024-01-09 17:34:00        NaT
8  Tomatoes      NaN 2024-01-01 13:03:00        NaT
9      Rice  43340.0 2024-02-04 12:40:00        NaT


In [ ]:
merged.to_csv("nigerian_market_merged_cleaned.csv", index=False)

from google.colab import files
files.download("nigerian_market_merged_cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Nigerian Market Data Wrangling Assignment

**Student:** Idorenyin Inwang  
**Date:** February 25, 2026  
**Assignment:** Data Wrangling & Cleaning – nigerian_market_prices.csv + nigerian_market_transactions.csv

## Objective
Clean and prepare two messy datasets for building a Nigerian market forecasting tool.  
Steps include:
- Loading and inspecting the data
- Handling missing values
- Converting data types (especially dates)
- Removing duplicates
- Merging the datasets
- Saving the final cleaned & merged file

## Datasets Used
- **nigerian_market_transactions_assignment.csv** – transaction records (transaction_id, date, market, product, quantity, revenue, etc.)
- **nigerian_market_prices_assignment.csv** – weekly market prices (product, price, date, market, etc.)

## Final Output
- Cleaned and merged dataset: `nigerian_market_merged_cleaned.csv`  
- Merge method: Left join on **product** + approximate **week start date**  
- Missing prices filled with product-level median

## Key Cleaning Decisions
- Removed currency symbol (₦) and commas from price column → converted to numeric
- Converted transaction_date and date columns to datetime
- Filled missing quantity/unit_price with medians (overall or per product)
- Recalculated total_amount where needed
- Used forward-fill or median for remaining gaps
- Dropped rows only when critical columns were missing after imputation

## Notebook Structure
1. Import libraries & upload files  
2. Load and inspect raw data  
3. Clean individual datasets (dates, missing values, types)  
4. Merge transactions with prices  
5. Handle post-merge missing values  
6. Final checks & save merged CSV

---

**Final merged file has been saved and downloaded.**
You can find it in the GitHub repository.

**GitHub Repo Link (to be added after upload):**  
https://github.com/idorenyininwang-byte/nigerian-market-data-wrangling-assignment

Submitted for review.

In [ ]:
# Confirm merged data is ready
print("Final merged shape:", merged.shape)
print("\nMissing values in final merged data:\n", merged.isnull().sum())
print("\nSample of merged data (first 5 rows):\n", merged.head())

Final merged shape: (1018, 17)

Missing values in final merged data:
 transaction_id        0
market_id_x           0
trader_id             0
trader_name_x         0
product               0
quantity              0
revenue               0
payment_method        0
transaction_date      0
week_start            0
market_id_y         949
market_name         949
state               949
price               397
unit                949
date                949
trader_name_y       949
dtype: int64

Sample of merged data (first 5 rows):
   transaction_id market_id_x trader_id trader_name_x   product  quantity  \
0        TXN0001      MKT005     TR005   ALHAJI MUSA      Gari         8   
1        TXN0002      MKT004     TR001        SADIYA     Beans        14   
2        TXN0002      MKT004     TR001        SADIYA     Beans        14   
3        TXN0003      MKT002     TR006   ALHAJI MUSA  Tomatoes         8   
4        TXN0004      MKT003     TR001        Sadiya     Beans        13   

   revenue p

In [ ]:
from google.colab import files

# Download the notebook (must match the name you just set)
files.download("Idorenyin_Nigerian_Market_Data_Wrangling.ipynb")

# Download the CSV again (just in case)
files.download("nigerian_market_merged_cleaned.csv")

FileNotFoundError: Cannot find file: Idorenyin_Nigerian_Market_Data_Wrangling.ipynb

In [ ]:
from google.colab import files

# Download the notebook (make sure you renamed it earlier)
files.download("Idorenyin_Nigerian_Market_Data_Wrangling.ipynb")  # change name if different

# Download the final CSV
files.download("nigerian_market_merged_cleaned.csv")

FileNotFoundError: Cannot find file: Idorenyin_Nigerian_Market_Data_Wrangling.ipynb

In [ ]:
from google.colab import files

# Use the name you just gave it
files.download("Idorenyin_Nigerian_Market_Data_Wrangling.ipynb")

# And the CSV (this one should still work)
files.download("nigerian_market_merged_cleaned.csv")

FileNotFoundError: Cannot find file: Idorenyin_Nigerian_Market_Data_Wrangling.ipynb

In [ ]:
from google.colab import files

# Download the notebook using the exact name you set
files.download("Idorenyin_Nigerian_Market_Data_Wrangling.ipynb")

# Download the cleaned CSV (should still be there)
files.download("nigerian_market_merged_cleaned.csv")

FileNotFoundError: Cannot find file: Idorenyin_Nigerian_Market_Data_Wrangling.ipynb

In [ ]:
!ls -lh *.ipynb

ls: cannot access '*.ipynb': No such file or directory


In [ ]:
from google.colab import files
files.download("Idorenyin_Nigerian_Market_Data_Wrangling.ipynb")

FileNotFoundError: Cannot find file: Idorenyin_Nigerian_Market_Data_Wrangling.ipynb